In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("C:/Users/melni/01-ecommerce-sales-analysis/data/raw")

files = {
    "customers":   "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {name: pd.read_csv(RAW / fname) for name, fname in files.items()}

for name, df in dfs.items():
    print(f"{name:22s} rows={df.shape[0]:>9,}  cols={df.shape[1]}")

customers              rows=   99,441  cols=5
geolocation            rows=1,000,163  cols=5
order_items            rows=  112,650  cols=7
payments               rows=  103,886  cols=5
reviews                rows=   99,224  cols=7
orders                 rows=   99,441  cols=8
products               rows=   32,951  cols=9
sellers                rows=    3,095  cols=4
category_translation   rows=       71  cols=2


In [2]:
for name, df in dfs.items():
    print("=" * 60)
    print(name.upper(), df.shape)
    print("-" * 60)
    print(df.dtypes.to_string())
    print("\nПропуски:")
    missing = df.isna().sum()
    print(missing[missing > 0].to_string() if missing.any() else "немає")
    print(f"\nПовних дублікатів рядків: {df.duplicated().sum()}")

CUSTOMERS (99441, 5)
------------------------------------------------------------
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object

Пропуски:
немає

Повних дублікатів рядків: 0
GEOLOCATION (1000163, 5)
------------------------------------------------------------
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object

Пропуски:
немає

Повних дублікатів рядків: 261831
ORDER_ITEMS (112650, 7)
------------------------------------------------------------
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64

Пропуски:
немає

Повних дублікатів рядків: 0
PAYMENTS (103886,

In [3]:
dfs["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders = dfs["orders"]
items = dfs["order_items"]
payments = dfs["payments"]
reviews = dfs["reviews"]
customers = dfs["customers"]
products = dfs["products"]
translation = dfs["category_translation"]

for col in ["order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col])

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col])

print("Готово. Типи дат у таблиці orders:")
print(orders.dtypes)

Готово. Типи дат у таблиці orders:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
print("Кількість замовлень за статусом:\n")
print(orders["order_status"].value_counts())

Кількість замовлень за статусом:

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
no_delivery = orders[orders["order_delivered_customer_date"].isna()]

print(f"Замовлень без дати доставки: {len(no_delivery)}\n")
print("Їхні статуси:")
print(no_delivery["order_status"].value_counts())

delivered_no_date = orders[(orders["order_status"] == "delivered")
                           & orders["order_delivered_customer_date"].isna()]
print(f"\nЗі статусом 'delivered', але без дати доставки: {len(delivered_no_date)}")

Замовлень без дати доставки: 2965

Їхні статуси:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Зі статусом 'delivered', але без дати доставки: 8


In [7]:
first = orders["order_purchase_timestamp"].min()
last = orders["order_purchase_timestamp"].max()
print(f"Перше замовлення: {first:%Y-%m-%d}")
print(f"Останнє замовлення: {last:%Y-%m-%d}\n")

monthly = (orders["order_purchase_timestamp"]
           .dt.to_period("M").value_counts().sort_index())
print("Замовлень по місяцях:\n")
print(monthly)

Перше замовлення: 2016-09-04
Останнє замовлення: 2018-10-17

Замовлень по місяцях:

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


In [9]:
bought = orders["order_purchase_timestamp"]

delivered_before_bought = (orders["order_delivered_customer_date"] < bought).sum()
approved_before_bought = (orders["order_approved_at"] < bought).sum()
delivered_before_carrier = (orders["order_delivered_customer_date"]
                            < orders["order_delivered_carrier_date"]).sum()

print(f"Доставлено раніше, ніж куплено: {delivered_before_bought}")
print(f"Схвалено раніше, ніж куплено: {approved_before_bought}")
print(f"Доставлено клієнту раніше, ніж передано перевізнику: {delivered_before_carrier}")

Доставлено раніше, ніж куплено: 0
Схвалено раніше, ніж куплено: 0
Доставлено клієнту раніше, ніж передано перевізнику: 23


In [8]:
print("ЦІНИ ТОВАРІВ")
print(f"  мінімум: {items['price'].min():.2f}")
print(f"  медіана: {items['price'].median():.2f}")
print(f"  максимум: {items['price'].max():.2f}")
print(f"  позицій з ціною 0 або менше: {(items['price'] <= 0).sum()}")

print("\nВАРТІСТЬ ДОСТАВКИ")
print(f"  від'ємних значень: {(items['freight_value'] < 0).sum()}")
print(f"  нульових (безкоштовна доставка): {(items['freight_value'] == 0).sum()}")

print("\nСПОСОБИ ОПЛАТИ")
print(payments["payment_type"].value_counts())

print(f"\nОплат із сумою 0 або менше: {(payments['payment_value'] <= 0).sum()}")

ЦІНИ ТОВАРІВ
  мінімум: 0.85
  медіана: 74.99
  максимум: 6735.00
  позицій з ціною 0 або менше: 0

ВАРТІСТЬ ДОСТАВКИ
  від'ємних значень: 0
  нульових (безкоштовна доставка): 383

СПОСОБИ ОПЛАТИ
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Оплат із сумою 0 або менше: 9


In [10]:
print("ЗВ'ЯЗКИ МІЖ ТАБЛИЦЯМИ")
print(f"  замовлень без жодного товару: {(~orders['order_id'].isin(items['order_id'])).sum()}")
print(f"  товарів, для яких немає замовлення: {(~items['order_id'].isin(orders['order_id'])).sum()}")
print(f"  замовлень без клієнта: {(~orders['customer_id'].isin(customers['customer_id'])).sum()}")

print("\nКЛІЄНТИ")
print(f"  унікальних customer_id: {customers['customer_id'].nunique()}")
print(f"  унікальних customer_unique_id: {customers['customer_unique_id'].nunique()}")

print("\nВІДГУКИ")
reviews_per_order = reviews.groupby("order_id").size()
print(f"  замовлень із кількома відгуками: {(reviews_per_order > 1).sum()}")
print(f"  замовлень без відгуків: {(~orders['order_id'].isin(reviews['order_id'])).sum()}")

print("\nКАТЕГОРІЇ")
known = products["product_category_name"].dropna()
untranslated = known[~known.isin(translation["product_category_name"])].unique()
print(f"  категорій без англійської назви: {list(untranslated)}")

ЗВ'ЯЗКИ МІЖ ТАБЛИЦЯМИ
  замовлень без жодного товару: 775
  товарів, для яких немає замовлення: 0
  замовлень без клієнта: 0

КЛІЄНТИ
  унікальних customer_id: 99441
  унікальних customer_unique_id: 96096

ВІДГУКИ
  замовлень із кількома відгуками: 547
  замовлень без відгуків: 768

КАТЕГОРІЇ
  категорій без англійської назви: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [11]:
PERIOD_START = "2017-01-01"
PERIOD_END = "2018-08-31"

orders_clean = orders.copy()

purchase = orders_clean["order_purchase_timestamp"]
delivered = orders_clean["order_delivered_customer_date"]
estimated = orders_clean["order_estimated_delivery_date"]

orders_clean["is_delivered"] = orders_clean["order_status"].eq("delivered")
orders_clean["purchase_month"] = purchase.dt.to_period("M").dt.to_timestamp()
orders_clean["delivery_days"] = ((delivered - purchase).dt.total_seconds() / 86400).round(1)
orders_clean["is_late"] = (delivered > estimated).astype("boolean").mask(delivered.isna())
orders_clean["in_analysis_period"] = (
    (purchase >= pd.Timestamp(PERIOD_START))
    & (purchase < pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1))
)

print(f"В аналізованому періоді: {orders_clean['in_analysis_period'].sum():,} із {len(orders_clean):,} замовлень")
orders_clean[["order_id", "order_status", "purchase_month", "delivery_days", "is_late", "in_analysis_period"]].head()

В аналізованому періоді: 99,092 із 99,441 замовлень


,order_id,order_status,purchase_month,delivery_days,is_late,in_analysis_period
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-01,8.4,False,True
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-01,13.8,False,True
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-01,9.4,False,True
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-01,13.2,False,True
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-01,2.9,False,True


In [12]:
products_clean = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

products_clean = products_clean.merge(translation, on="product_category_name", how="left")
products_clean["category"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"])
    .fillna("unknown")
)
products_clean = products_clean.drop(columns=["product_category_name_english"])

print(f"Товарів: {len(products_clean):,}")
print(f"Категорій (разом з unknown): {products_clean['category'].nunique()}")
print(f"Товарів у категорії unknown: {(products_clean['category'] == 'unknown').sum()}")

Товарів: 32,951
Категорій (разом з unknown): 74
Товарів у категорії unknown: 610


In [13]:
reviews_clean = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates("order_id", keep="last")
    [["order_id", "review_score", "review_creation_date", "review_answer_timestamp"]]
)

print(f"Було відгуків: {len(reviews):,}")
print(f"Стало (по одному на замовлення): {len(reviews_clean):,}")
print(f"Унікальних замовлень: {reviews_clean['order_id'].nunique():,}")

Було відгуків: 99,224
Стало (по одному на замовлення): 98,673
Унікальних замовлень: 98,673


In [14]:
geo = dfs["geolocation"]

geo_clean = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "median"),
         lng=("geolocation_lng", "median"),
         city=("geolocation_city", "first"),
         state=("geolocation_state", "first"))
    .reset_index()
    .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})
)

print(f"Було рядків: {len(geo):,}")
print(f"Стало: {len(geo_clean):,}")

Було рядків: 1,000,163
Стало: 19,015


In [16]:
OUT = Path("C:/Users/melni/01-ecommerce-sales-analysis/data/processed")
OUT.mkdir(parents=True, exist_ok=True)

tables = {
    "orders": orders_clean,
    "order_items": items,
    "payments": payments,
    "reviews": reviews_clean,
    "customers": customers,
    "products": products_clean,
    "sellers": dfs["sellers"],
    "geolocation": geo_clean,
    "categories": translation,
}

for name, df in tables.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
    print(f"{name:12s} {len(df):>8,} рядків збережено")

orders         99,441 рядків збережено
order_items   112,650 рядків збережено
payments      103,886 рядків збережено
reviews        98,673 рядків збережено
customers      99,441 рядків збережено
products       32,951 рядків збережено
sellers         3,095 рядків збережено
geolocation    19,015 рядків збережено
categories         71 рядків збережено


In [17]:
check = pd.read_csv(OUT / "orders.csv")
print("Замовлень у чистому файлі:", len(check), "(було 99 441)")
print("Порожніх order_id:", check["order_id"].isna().sum())
print("Дублікатів order_id:", check["order_id"].duplicated().sum())

print("\nДоставлених замовлень:", check["is_delivered"].sum())
print("У періоді аналізу:", check["in_analysis_period"].sum())

Замовлень у чистому файлі: 99441 (було 99 441)
Порожніх order_id: 0
Дублікатів order_id: 0

Доставлених замовлень: 96478
У періоді аналізу: 99092
